# RAG System — pgvector + SQL Agent
End-to-end pipeline: natural language → pgvector retrieval → SQL agent → JSON output

## 1. Setup

In [1]:
import sys
sys.path.append("..")

In [2]:
# pgvector RAG — run once to create table + seed all 10 business rules
from pgvector_rag import setup_pgvector_table, seed_business_rules, retrieve_business_rules

setup_pgvector_table()
seed_business_rules()

c:\Users\Admin\Desktop\Coding\RAG System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ pgvector table + index created (or already exists).
  ⏭  Skipping rule_discount_001 (already seeded)
  ⏭  Skipping rule_discount_002 (already seeded)
  ⏭  Skipping rule_discount_003 (already seeded)
  ⏭  Skipping rule_join_001 (already seeded)
  ⏭  Skipping rule_join_002 (already seeded)
  ⏭  Skipping rule_join_003 (already seeded)
  ⏭  Skipping rule_agg_001 (already seeded)
  ⏭  Skipping rule_agg_002 (already seeded)
  ⏭  Skipping rule_agg_003 (already seeded)
  ⏭  Skipping rule_agg_004 (already seeded)
Done seeding business rules into pgvector.


## 2. LLM + Database + Toolkit

In [3]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="openai/gpt-oss-120b",
    temperature=0,
    top_p=1
)

c:\Users\Admin\Desktop\Coding\RAG System\venv\Lib\site-packages\pydantic\main.py:250: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [4]:
from langchain_community.utilities import SQLDatabase
from sql_alchemy.engine import engine

db = SQLDatabase(engine, ignore_tables=["business_rules", "rag_documents"])
print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())

Connection successful!
Dialect: postgresql
Tables: ['discounts', 't_shirts']


In [5]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

## 3. RAG-Powered SQL Agent

For each question:
1. `retrieve_business_rules()` embeds the question and does cosine search in pgvector
2. Retrieved rules are injected into the SQL agent prefix
3. The SQL agent generates a JSON-locked response

In [10]:
from langchain_community.agent_toolkits.sql.base import create_sql_agent

def run_rag_agent(question: str) -> str:
    """
    Full pipeline:
      1. Retrieve relevant business rules from pgvector (384-dim cosine search)
      2. Inject rules into SQL agent prefix
      3. Run SQL agent -> returns JSON with sql, metric, assumptions
    """

    # Step 1: pgvector semantic retrieval
    retrieved_rules = retrieve_business_rules(question, top_k=3)
    print(f"\nRetrieved {len(retrieved_rules)} rules from pgvector:")
    for r in retrieved_rules:
        print(f"  [{r['similarity']:.3f}] {r['tier']}: {r['rule'][:80]}...")

    # Step 2: Format retrieved rules as context block
    if retrieved_rules:
        rag_context = "\n".join(
            [str(i) + ". [" + r["tier"] + "] " + r["rule"] for i, r in enumerate(retrieved_rules, 1)]
        )
    else:
        rag_context = "No specific business rules retrieved. Use general SQL best practices."

    # Step 3: Build prefix using string concatenation (avoids .format() brace conflicts)
    prefix = (
        "You are an agent designed to interact with a SQL database.\n"
        "Given an input question, create a syntactically correct postgresql query to run,\n"
        "then look at the results of the query and return the answer.\n\n"
        "You MUST double check your query before executing it. If you get an error while "
        "executing a query, rewrite the query and try again.\n\n"
        "DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.\n\n"
        "To start you should ALWAYS look at the tables in the database to see what you "
        "can query. Do NOT skip this step. Then query the schema of the most relevant tables.\n\n"
        "RETRIEVED BUSINESS RULES (via pgvector semantic search - 384-dim cosine similarity):\n\n"
        + rag_context +
        "\n\nIMPORTANT:\n"
        "- The above rules were retrieved specifically for this question. Apply them.\n"
        "- Do NOT hallucinate rules not present here.\n\n"
        "IMPORTANT DATABASE RULES:\n"
        "- Column values such as brand, color, and size are ENUMs and are CASE-SENSITIVE.\n"
        "- You MUST inspect the table schema before filtering on ENUM columns.\n"
        "- You MUST match user input to the closest valid ENUM value from the database.\n"
        "  Example: white -> White, nike -> Nike, extra small -> XS.\n"
        "- If a query returns zero rows, re-check casing and ENUM values before answering.\n"
        "- Do NOT guess. Do NOT generalize. Do NOT switch brands or sizes.\n"
        "- Use COALESCE(discount, 0) whenever discount may be NULL.\n"
        "- Use LEFT JOIN when combining tables unless INNER JOIN is explicitly justified.\n\n"
        "OUTPUT FORMAT (STRICT):\n"
        "You MUST respond in a JSON like format. No explanations outside JSON. No markdown. You do not require any JSON tool for this, just format your answer in this format.\n"
        "- sql: a single SQL query string (read-only SELECT only)\n"
        "- metric: the primary metric being computed\n"
        "- assumptions: a list of assumptions made while forming the SQL\n"
        "- answer: the actual computed result/value from running the SQL\n"
    )

    # Step 4: Run the SQL agent
    agent = create_sql_agent(
        llm=llm,
        toolkit=toolkit,
        verbose=True,
        agent_type="openai-tools",
        handle_parsing_errors=True,
        prefix=prefix
    )

    response = agent.invoke({"input": question})
    return response["output"]

## 4. Run Queries — 3 Complexity Tiers

### Tier 1 — Discounted Revenue

In [11]:
question = "If we sell all Levi's T-shirts today with discounts applied, how much revenue will we generate?"
result = run_rag_agent(question)
print("\nFinal Output:")
print(result)


Retrieved 3 rules from pgvector:
  [0.406] tier1_discounted_revenue: Revenue without discounts = price × stock_quantity. Use this when the question d...
  [0.370] tier1_discounted_revenue: Post-discount revenue = price × quantity × (1 - discount_rate). discount_rate is...
  [0.353] tier1_discounted_revenue: Discounts are optional per product. Revenue must include both discounted and non...


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


discounts, t_shirts
Invoking: `sql_db_schema` with `{'table_names': 't_shirts'}`



CREATE TABLE t_shirts (
	t_shirt_id SERIAL NOT NULL, 
	brand brand_enum NOT NULL, 
	color color_enum NOT NULL, 
	size size_enum NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	CONSTRAINT t_shirts_pkey PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_brand_color_size_key UNIQUE NULLS DISTINCT (brand, color, size), 
	CONSTRAINT t_shirts_price_check CHECK (price >= 10 AND price <= 50)
)

/*
3 rows from

### Tier 2 — Optional Joins

In [12]:
question = "Show me all brands and how many discounts each brand has, including brands with no discounts."
result = run_rag_agent(question)
print("\nFinal Output:")
print(result)


Retrieved 3 rules from pgvector:
  [0.524] tier1_discounted_revenue: Discounts are optional per product. Revenue must include both discounted and non...
  [0.486] tier1_discounted_revenue: Revenue without discounts = price × stock_quantity. Use this when the question d...
  [0.421] tier3_aggregations: When ranking or comparing brands/categories, always use GROUP BY and aggregate f...


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


discounts, t_shirts
Invoking: `sql_db_schema` with `{'table_names': 'discounts, t_shirts'}`



CREATE TABLE discounts (
	discount_id SERIAL NOT NULL, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount NUMERIC(5, 2), 
	CONSTRAINT discounts_pkey PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_t_shirt_id_fkey FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id) ON DELETE CASCADE, 
	CONSTRAINT discounts_pct_discount_check CHECK (pct_discount >= 0::numeric AND pct_discount <= 100::numeric)
)

/*
3 rows from discounts tabl

### Tier 3 — Aggregations

In [13]:
question = "Which brand has the highest total inventory value across all t-shirts?"
result = run_rag_agent(question)
print("\nFinal Output:")
print(result)


Retrieved 0 rules from pgvector:


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


discounts, t_shirts
Invoking: `sql_db_schema` with `{'table_names': 't_shirts'}`



CREATE TABLE t_shirts (
	t_shirt_id SERIAL NOT NULL, 
	brand brand_enum NOT NULL, 
	color color_enum NOT NULL, 
	size size_enum NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	CONSTRAINT t_shirts_pkey PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_brand_color_size_key UNIQUE NULLS DISTINCT (brand, color, size), 
	CONSTRAINT t_shirts_price_check CHECK (price >= 10 AND price <= 50)
)

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock_quantity
1	Adidas	White	L	18	53
2	Adidas	Black	M	23	30
3	Adidas	Black	L	43	78
*/
Invoking: `sql_db_query` with `{'query': 'SELECT brand, SUM(price * stock_quantity) AS total_value FROM t_shirts GROUP BY brand ORDER BY total_value DESC LIMIT 1;'}`


[('Van Huesen', 25076)]{
  "sql": "SELECT brand, SUM(price * stock_qua